# Manifest And Dataset Audit

Audit the canonical processed dataset summary by default, while still supporting alternate raw manifests.

In [ ]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_01_MANIFEST_AND_DATASET_AUDIT_CELL_PROGRESS_1 = start_notebook_cell_progress('01_manifest_and_dataset_audit.ipynb', 'Load shared setup', total_steps=1)

import json
from pathlib import Path

import pandas as pd

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

finish_notebook_cell_progress(NB_01_MANIFEST_AND_DATASET_AUDIT_CELL_PROGRESS_1)


In [ ]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_01_MANIFEST_AND_DATASET_AUDIT_CELL_PROGRESS_2 = start_notebook_cell_progress('01_manifest_and_dataset_audit.ipynb', 'Define audit helpers', total_steps=1)

LABEL_ALIASES = {
    'fresh': 'fresh',
    'not fresh': 'not fresh',
    'not_fresh': 'not fresh',
    'notfresh': 'not fresh',
    'spoiled': 'spoiled',
}


def normalize_label(value: str) -> str:
    normalized = ' '.join(str(value).strip().lower().replace('_', ' ').split())
    if normalized not in LABEL_ALIASES:
        raise ValueError(f'Unsupported label: {value!r}')
    return LABEL_ALIASES[normalized]


def load_manifest_table(manifest_path: Path) -> pd.DataFrame:
    suffix = manifest_path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(manifest_path, dtype=str).fillna('')
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(manifest_path, dtype=str).fillna('')
    raise ValueError(f'Unsupported manifest file: {manifest_path}')


def build_processed_local_path(row: pd.Series, processed_root: Path) -> Path:
    return processed_root / f"sample {str(row['sample_number']).strip()}" / str(row['label']).strip() / str(row['image_file_name']).strip()


def resolve_alternate_image_path(row: pd.Series, images_root: Path) -> Path:
    raw_value = str(row.get('image_name', '')).strip()
    candidate = Path(raw_value)
    if candidate.is_absolute():
        return candidate
    return images_root / candidate


def audit_manifest(manifest_path: Path, processed_root: Path, images_root: Path) -> pd.DataFrame:
    manifest_df = load_manifest_table(manifest_path)
    canonical_columns = {'image_file_name', 'label', 'sample_number', 'sample_id'}

    if canonical_columns.issubset(set(manifest_df.columns)):
        audited_df = manifest_df.copy()
        audited_df['label'] = audited_df['label'].map(normalize_label)
        audited_df['image_name'] = audited_df['image_file_name']
        audited_df['sample_number'] = audited_df['sample_number'].astype(str).str.strip()
        audited_df['sample_id'] = audited_df['sample_id'].astype(str).str.strip()
        audited_df['local_image_path'] = audited_df.apply(
            lambda row: str(build_processed_local_path(row, processed_root)),
            axis=1,
        )
        audited_df['source_manifest_type'] = 'processed_summary'
    else:
        required_columns = {'image_name', 'label'}
        missing = required_columns - set(manifest_df.columns)
        if missing:
            raise ValueError(f'Missing required manifest columns: {sorted(missing)}')

        audited_df = manifest_df.copy()
        audited_df['label'] = audited_df['label'].map(normalize_label)
        audited_df['image_name'] = audited_df['image_name'].astype(str).str.strip()
        audited_df['image_file_name'] = audited_df['image_name']
        audited_df['sample_number'] = audited_df.get('sample_number', '').astype(str).str.strip()
        audited_df['sample_id'] = audited_df.get('sample_id', '').astype(str).str.strip()
        audited_df['local_image_path'] = audited_df.apply(
            lambda row: str(resolve_alternate_image_path(row, images_root)),
            axis=1,
        )
        audited_df['source_manifest_type'] = 'raw_manifest'

    missing_paths = [path for path in audited_df['local_image_path'].map(Path) if not path.exists()]
    if missing_paths:
        missing_preview = [str(path) for path in missing_paths[:5]]
        raise FileNotFoundError(f'Unable to resolve {len(missing_paths)} image paths: {missing_preview}')

    return audited_df

finish_notebook_cell_progress(NB_01_MANIFEST_AND_DATASET_AUDIT_CELL_PROGRESS_2)


In [ ]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_01_MANIFEST_AND_DATASET_AUDIT_CELL_PROGRESS_3 = start_notebook_cell_progress('01_manifest_and_dataset_audit.ipynb', 'Audit manifest', total_steps=1)

MANIFEST_PATH = Path(str(override('MANIFEST_PATH', CANONICAL_PROCESSING_SUMMARY_PATH)))
IMAGES_ROOT = Path(str(override('IMAGES_ROOT', RAW_DATA_ROOT)))
AUDITED_MANIFEST_PATH = Path(str(override('AUDITED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'audited_manifest.csv')))

ensure_dir(AUDITED_MANIFEST_PATH.parent)
audited_df = audit_manifest(MANIFEST_PATH, PROCESSED_ROI_ROOT, IMAGES_ROOT)
audited_df.to_csv(AUDITED_MANIFEST_PATH, index=False)

print(f'Audited rows: {len(audited_df)}')
print(audited_df.groupby(['label']).size().to_string())

finish_notebook_cell_progress(NB_01_MANIFEST_AND_DATASET_AUDIT_CELL_PROGRESS_3)
